In [4]:
%pip install openeo

  Using cached openeo-0.51.0-py3-none-any.whl.metadata (8.9 kB)
  Using cached shapely-2.1.2-cp312-cp312-win_amd64.whl.metadata (7.1 kB)
  Using cached xarray-2025.1.1-py3-none-any.whl.metadata (11 kB)
  Using cached pandas-2.3.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached pystac-1.15.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached deprecated-1.3.1-py2.py3-none-any.whl.metadata (5.9 kB)
  Using cached oschmod-0.3.12-py2.py3-none-any.whl.metadata (10.0 kB)
  Using cached geopandas-1.1.4-py3-none-any.whl.metadata (2.3 kB)
  Using cached wrapt-2.4.0-cp312-cp312-win_amd64.whl.metadata (7.6 kB)
  Using cached pyogrio-0.13.0-cp311-abi3-win_amd64.whl.metadata (6.0 kB)
  Using cached pyproj-3.7.2-cp312-cp312-win_amd64.whl.metadata (31 kB)
  Using cached pytz-2026.3.post1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached pystac_core-1.15.2-py3-none-any.whl.metadata (1.3 kB)
  Using cached pystac_ext_classification-2.0.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached pyst


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import openeo

connection = openeo.connect(
    "openeo.dataspace.copernicus.eu"
)

print(connection)

<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with NullAuth>


In [3]:
connection.authenticate_oidc()

Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

In [4]:
ngawi_bbox = {
    "west": 111.118446,
    "south": -7.620417,
    "east": 111.671219,
    "north": -7.244574
}

# AOI (Area of Interest) dalam format polygon GeoJSON
aoi = {
    "type": "Polygon",
    "coordinates": [
        [
            [111.118446, -7.620417],
            [111.118446, -7.244574],
            [111.671219, -7.244574],
            [111.671219, -7.620417],
            [111.118446, -7.620417],
        ]
    ]
}

tanggal_mulai = "2025-08-25"
tanggal_selesai = "2026-08-25"

# Daftar polutan yang ingin diambil
daftar_polutan = ["NO2", "SO2", "O3", "CO"]

for polutan in daftar_polutan:
    print(f"Memproses polutan: {polutan}")

    s5p = connection.load_collection(
        "SENTINEL_5P_L2",
        temporal_extent=[tanggal_mulai, tanggal_selesai],
        spatial_extent=ngawi_bbox,
        bands=[polutan],
    )

    # Agregasi harian agar tidak ada lebih dari satu data per hari
    s5p_daily = s5p.aggregate_temporal_period(reducer="mean", period="day")

    # Agregasi spasial agar seluruh grid wilayah Ngawi dirata-rata jadi satu nilai
    s5p_aoi = s5p_daily.aggregate_spatial(reducer="mean", geometries=aoi)

    # Simpan hasil sebagai CSV
    result = s5p_aoi.save_result(format="CSV")

    # Jalankan job dan tunggu sampai selesai
    job = result.create_job(title=f"s5p_{polutan.lower()}_ngawi")
    job.start_and_wait()

    # Download hasil
    job.get_results().download_files(f"output_{polutan.lower()}_ngawi")

    print(f"Selesai: {polutan}\n")

Memproses polutan: NO2
0:00:00 Job 'j-2609130702374c0d9a1e2081d13bd3eb': send 'start'
0:00:03 Job 'j-2609130702374c0d9a1e2081d13bd3eb': queued (progress 0%)
0:00:09 Job 'j-2609130702374c0d9a1e2081d13bd3eb': queued (progress 0%)
0:00:15 Job 'j-2609130702374c0d9a1e2081d13bd3eb': queued (progress 0%)
0:00:24 Job 'j-2609130702374c0d9a1e2081d13bd3eb': queued (progress 0%)
0:00:34 Job 'j-2609130702374c0d9a1e2081d13bd3eb': running (progress N/A)
0:00:46 Job 'j-2609130702374c0d9a1e2081d13bd3eb': running (progress N/A)
0:01:02 Job 'j-2609130702374c0d9a1e2081d13bd3eb': running (progress N/A)
0:01:22 Job 'j-2609130702374c0d9a1e2081d13bd3eb': running (progress N/A)
0:01:46 Job 'j-2609130702374c0d9a1e2081d13bd3eb': running (progress N/A)
0:02:16 Job 'j-2609130702374c0d9a1e2081d13bd3eb': running (progress N/A)
0:02:53 Job 'j-2609130702374c0d9a1e2081d13bd3eb': running (progress N/A)
0:03:40 Job 'j-2609130702374c0d9a1e2081d13bd3eb': running (progress N/A)
0:04:39 Job 'j-2609130702374c0d9a1e2081d13bd3e

In [ ]:
import pandas as pd

daftar_file = {
    "NO2": "output_no2_ngawi/timeseries.csv",
    "SO2": "output_so2_ngawi/timeseries.csv",
    "O3": "output_o3_ngawi/timeseries.csv",
    "CO": "output_co_ngawi/timeseries.csv",
}

dataframes = {}

for polutan, path in daftar_file.items():
    print(f"===== {polutan} =====")
    df = pd.read_csv(path)
    dataframes[polutan] = df

    print("Shape:", df.shape)
    print("\nInfo:")
    df.info()
    print("\nCuplikan data:")
    print(df.head())
    print("\nJumlah nilai unik kolom date:", df['date'].nunique() if 'date' in df.columns else "kolom 'date' tidak ditemukan")
    print("\n")

===== NO2 =====
Shape: (366, 3)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           366 non-null    object 
 1   feature_index  366 non-null    int64  
 2   NO2            300 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 8.7+ KB

Cuplikan data:
                       date  feature_index       NO2
0  2026-07-21T00:00:00.000Z              0  0.000024
1  2026-07-15T00:00:00.000Z              0  0.000033
2  2026-07-20T00:00:00.000Z              0  0.000030
3  2026-07-17T00:00:00.000Z              0  0.000035
4  2026-07-16T00:00:00.000Z              0  0.000031

Jumlah nilai unik kolom date: 366


===== SO2 =====
Shape: (366, 3)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 366 entries, 0 to 365
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------      

In [24]:
daftar_file = {
    "NO2": "output_no2_ngawi/timeseries.csv",
    "SO2": "output_so2_ngawi/timeseries.csv",
    "O3": "output_o3_ngawi/timeseries.csv",
    "CO": "output_co_ngawi/timeseries.csv",
}

for polutan, path in daftar_file.items():
    df = pd.read_csv(path)

    # Pastikan kolom tanggal valid & buang info timezone
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["date"] = df["date"].dt.strftime("%Y-%m-%d")

    # Buang kolom feature_index, hanya simpan date & nilai polutan
    new_df = pd.DataFrame({
        "date": df["date"],
        polutan: df[polutan]
    })

    new_df.to_csv(f"{polutan}_Timeseries.csv", index=False)
    print(f"Selesai normalisasi: {polutan} -> {polutan}_Timeseries.csv")

Selesai normalisasi: NO2 -> NO2_Timeseries.csv
Selesai normalisasi: SO2 -> SO2_Timeseries.csv
Selesai normalisasi: O3 -> O3_Timeseries.csv
Selesai normalisasi: CO -> CO_Timeseries.csv


In [25]:
import pandas as pd

daftar_polutan = ["NO2", "SO2", "O3", "CO"]

start_date = "2025-08-25"
end_date   = "2026-08-25"
full_range = pd.date_range(start=start_date, end=end_date, freq='D')

for polutan in daftar_polutan:
    df = pd.read_csv(f"{polutan}_Timeseries.csv")
    df['date'] = pd.to_datetime(df['date'])

    # Cek tanggal yang hilang
    missing_dates = full_range.difference(df['date'])

    print(f"===== {polutan} =====")
    print(f"Jumlah hari missing: {len(missing_dates)}")
    if len(missing_dates) > 0:
        print("Daftar tanggal missing:")
        print(missing_dates)
    print()

===== NO2 =====
Jumlah hari missing: 1
Daftar tanggal missing:
DatetimeIndex(['2026-08-25'], dtype='datetime64[ns]', freq='D')

===== SO2 =====
Jumlah hari missing: 1
Daftar tanggal missing:
DatetimeIndex(['2026-08-25'], dtype='datetime64[ns]', freq='D')

===== O3 =====
Jumlah hari missing: 1
Daftar tanggal missing:
DatetimeIndex(['2026-08-25'], dtype='datetime64[ns]', freq='D')

===== CO =====
Jumlah hari missing: 1
Daftar tanggal missing:
DatetimeIndex(['2026-08-25'], dtype='datetime64[ns]', freq='D')



In [26]:
import pandas as pd

daftar_polutan = ["NO2", "SO2", "O3", "CO"]

for polutan in daftar_polutan:
    df = pd.read_csv(f"{polutan}_Timeseries.csv")
    
    jumlah_missing = df[polutan].isna().sum()
    total_baris = len(df)
    persentase = (jumlah_missing / total_baris) * 100

    print(f"===== {polutan} =====")
    print(f"Jumlah nilai kosong (NaN): {jumlah_missing} dari {total_baris} baris ({persentase:.2f}%)")
    print()

===== NO2 =====
Jumlah nilai kosong (NaN): 66 dari 366 baris (18.03%)

===== SO2 =====
Jumlah nilai kosong (NaN): 41 dari 366 baris (11.20%)

===== O3 =====
Jumlah nilai kosong (NaN): 5 dari 366 baris (1.37%)

===== CO =====
Jumlah nilai kosong (NaN): 40 dari 366 baris (10.93%)



In [28]:
import pandas as pd
from sklearn.ensemble import IsolationForest

daftar_polutan = ["NO2", "SO2", "O3", "CO"]

for polutan in daftar_polutan:
    df = pd.read_csv(f"{polutan}_Timeseries.csv")
    df_clean = df.dropna(subset=[polutan]).copy()

    model = IsolationForest(contamination=0.05, random_state=42)  # contamination 0.05 = 5%
    pred = model.fit_predict(df_clean[[polutan]])

    # Nilai -1 merepresentasikan outlier
    df_clean["is_outlier"] = pred
    jumlah_outlier = (pred == -1).sum()

    print(f"===== {polutan} =====")
    print(f"Jumlah baris valid (non-NaN): {len(df_clean)}")
    print(f"Jumlah outlier: {jumlah_outlier}")
    print("Contoh baris outlier:")
    print(df_clean[df_clean["is_outlier"] == -1].head())
    print()

===== NO2 =====
Jumlah baris valid (non-NaN): 300
Jumlah outlier: 15
Contoh baris outlier:
           date       NO2  is_outlier
26   2026-02-11  0.000007          -1
30   2026-02-08  0.000046          -1
47   2026-01-24  0.000005          -1
106  2025-12-16  0.000009          -1
114  2026-03-21  0.000011          -1

===== SO2 =====
Jumlah baris valid (non-NaN): 325
Jumlah outlier: 17
Contoh baris outlier:
           date       SO2  is_outlier
25   2025-11-19 -0.000805          -1
54   2026-06-08  0.000298          -1
63   2025-09-01  0.000394          -1
79   2025-10-24 -0.000194          -1
105  2026-02-23 -0.000417          -1

===== O3 =====
Jumlah baris valid (non-NaN): 361
Jumlah outlier: 18
Contoh baris outlier:
          date        O3  is_outlier
14  2026-02-20  0.110673          -1
25  2026-08-21  0.123320          -1
26  2026-08-18  0.121799          -1
29  2026-08-22  0.121267          -1
30  2026-08-17  0.120814          -1

===== CO =====
Jumlah baris valid (non-NaN): 32

In [29]:
import pandas as pd

df_no2 = pd.read_csv("NO2_Timeseries.csv")
df_so2 = pd.read_csv("SO2_Timeseries.csv")
df_o3  = pd.read_csv("O3_Timeseries.csv")
df_co  = pd.read_csv("CO_Timeseries.csv")

# Gabungkan berdasarkan kolom date (merge, bukan asumsi urutan baris sama)
df_merged = df_no2.merge(df_so2, on="date", how="outer") \
                   .merge(df_o3, on="date", how="outer") \
                   .merge(df_co, on="date", how="outer")

# Urutkan berdasarkan tanggal
df_merged = df_merged.sort_values("date").reset_index(drop=True)

df_merged.to_csv("Polutan_Ngawi.csv", index=False)

print("Shape hasil gabungan:", df_merged.shape)
print(df_merged.head())

Shape hasil gabungan: (366, 5)
         date       NO2       SO2        O3        CO
0  2025-08-24       NaN       NaN       NaN       NaN
1  2025-08-25  0.000022  0.000017  0.116711  0.031344
2  2025-08-26  0.000035 -0.000082  0.115135       NaN
3  2025-08-27  0.000019  0.000085  0.116175  0.023245
4  2025-08-28  0.000014  0.000074  0.113984  0.026982
